In [ ]:
# ─── Environment Setup (do not edit) ────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401

        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab": PROJECT_ROOT / ".env.colab",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac": PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    print("⚠️  WARNING: Could not detect platform. Falling back to .env (local).")
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    print(
        f"⚠️  WARNING: Detected /workspace but VAST_CONTAINERLABEL is not set. Assuming Vast.ai."
    )
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv

if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    print(f"⚠️  WARNING: Expected env file not found: {_env_file}")
    if _fallback.exists():
        print(f"   Falling back to: {_fallback}")
        _env_file = _fallback
    else:
        raise FileNotFoundError(
            f"No .env file found. Copy the correct example file:\n"
            f"  cp .env.{PLATFORM or 'mac'}.example .env.{PLATFORM or 'mac'}"
        )

load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()

print(f"✅ Platform : {PLATFORM or 'unknown (local fallback)'}")
print(f"✅ Env file : {_env_file}")
print(f"✅ DATA_ROOT: {DATA_ROOT}")
print(f"{'✅' if DATA_ROOT.exists() else '❌'} Path exists: {DATA_ROOT.exists()}")
# ─────────────────────────────────────────────────────────────────────────────

# AI-Art Signal Fusion — Stage 3.6

Loads the Stage 3.5 checkpoint (frozen CLIP ViT-L/14 + classifier head trained on
real-vs-AI-generated art) and applies it to the images already used by the main COOLANT
pipeline (`coolant_{train,dev,test}.h5`, produced by `02_preprocessing.ipynb`).

For each COOLANT image, this computes a scalar `ai_generated_score` in `[0, 1]` — how much
the frozen classifier thinks the image looks AI-generated. This is exported as an auxiliary
signal keyed by `article_id`, kept **separate** from the COOLANT ResNet50 image features
(2048-dim) rather than fused into them — no COOLANT architecture change required.

```
Input:  training/checkpoints_ai_art/<run>/checkpoint_manifest.json  (Stage 3.5 output)
        processed_data/hdf5/coolant_{train,dev,test}.h5             (image_paths dataset)
Output: training/ai_art_signal/ai_art_signal_{train,dev,test}.npz
```

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
# All tunable parameters and paths live here. Edit this cell only.
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

CONFIG = {
    "paths": {
        # Set to a specific run dir's checkpoint_manifest.json, or None to auto-pick
        # the most recently modified manifest under checkpoints_ai_art/.
        "stage35_manifest": None,
        "checkpoints_ai_art_root": DATA_ROOT / "training" / "checkpoints_ai_art",
        "hdf5_dir": DATA_ROOT / "processed_data" / "hdf5",
        "output_dir": DATA_ROOT / "training" / "ai_art_signal",
    },
    "data": {
        "splits": ["train", "dev", "test"],
    },
    "inference": {
        "batch_size": 32,
    },
    "safety": {
        "smoke_test": False,
        "smoke_n_samples": 20,
        "auto_install_deps": False,
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"HDF5 dir:     {CONFIG['paths']['hdf5_dir']}")
print(f"Output dir:   {CONFIG['paths']['output_dir']}")

In [ ]:
# ── DEPENDENCY PREFLIGHT ────────────────────────────────────────────────────
import importlib, sys

_required = {
    "torch": "torch",
    "torchvision": "torchvision",
    "transformers": "transformers",
    "h5py": "h5py",
    "numpy": "numpy",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "PIL": "pillow",
}

_missing = []
for mod, pkg in _required.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        _missing.append(pkg)

if _missing:
    if CONFIG["safety"]["auto_install_deps"]:
        import subprocess

        subprocess.check_call([sys.executable, "-m", "pip", "install"] + _missing)
        print(f"Installed: {_missing}")
    else:
        print(f"Missing packages: {_missing}")
        print("Install with:  pip install " + " ".join(_missing))
        raise RuntimeError(f"Missing packages: {_missing}.")
else:
    print("All dependencies satisfied.")

In [ ]:
# ── IMPORTS AND SETUP ───────────────────────────────────────────────────────
import sys, os, gc, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import h5py
from tqdm.auto import tqdm
from PIL import Image

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.image_preprocessing import ImagePreprocessor

CONFIG["paths"]["output_dir"].mkdir(parents=True, exist_ok=True)

print(f"PyTorch : {torch.__version__}")


def select_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"Device: cuda ({torch.cuda.get_device_name(0)}, {mem:.1f} GB)")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("Device: mps (Apple Silicon). For full run, prefer a CUDA GPU.")
    else:
        dev = torch.device("cpu")
        print("Device: cpu. Use smoke_test=True for local validation.")
    return dev


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Seed: {seed}")


DEVICE = select_device()
seed_everything(42)

## Step 1: Load Stage 3.5 Checkpoint Manifest

Auto-picks the most recently modified `checkpoint_manifest.json` under
`checkpoints_ai_art/` unless `CONFIG['paths']['stage35_manifest']` is set explicitly.

In [ ]:
def find_latest_manifest(root):
    manifests = sorted(
        Path(root).glob("*/checkpoint_manifest.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not manifests:
        raise FileNotFoundError(
            f"No checkpoint_manifest.json found under {root}. "
            "Run 03.5_ai_art_detection_training.ipynb first."
        )
    return manifests[0]


manifest_path = CONFIG["paths"]["stage35_manifest"] or find_latest_manifest(
    CONFIG["paths"]["checkpoints_ai_art_root"]
)
manifest_path = Path(manifest_path)

with open(manifest_path) as f:
    stage35_manifest = json.load(f)

print(f"Manifest: {manifest_path}")
print(f"  backbone       : {stage35_manifest['backbone']}")
print(f"  feature_dim    : {stage35_manifest['feature_dim']}")
print(f"  best_epoch     : {stage35_manifest['best_epoch']}")
print(f"  best_metrics   : {stage35_manifest['best_metrics']}")

checkpoint_path = Path(stage35_manifest["best_checkpoint_path"])
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint referenced by manifest not found: {checkpoint_path}")

## Step 2: Rebuild and Load the Classifier Head

In [ ]:
class AIArtClassifierHead(nn.Module):
    """Binary classifier head on frozen CLIP image features. 0=Real, 1=AI-generated."""

    def __init__(self, feature_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, features):
        return self.net(features).squeeze(-1)  # [B] logit


_ckpt = torch.load(checkpoint_path, map_location=DEVICE)

classifier_head = AIArtClassifierHead(
    feature_dim=_ckpt["feature_dim"],
    hidden_dim=_ckpt["hidden_dim"],
    dropout=_ckpt["dropout"],
).to(DEVICE)
classifier_head.load_state_dict(_ckpt["model_state_dict"])
classifier_head.eval()

image_preprocessor = ImagePreprocessor(model_name=stage35_manifest["backbone"], device=str(DEVICE))

print(f"Loaded classifier head (epoch {_ckpt['epoch']}) + {stage35_manifest['backbone']} backbone.")

## Step 3: Score COOLANT Images

For each split's HDF5 file, reads `image_paths` + `article_ids`, extracts CLIP features per
image, and runs them through the frozen classifier head to get `ai_generated_score = sigmoid(logit)`.

In [ ]:
def score_coolant_split(split, config, preprocessor, head, device):
    hdf5_path = config["paths"]["hdf5_dir"] / f"coolant_{split}.h5"
    if not hdf5_path.exists():
        print(f"  [skip] {split}: {hdf5_path} not found")
        return None

    with h5py.File(hdf5_path, "r") as f:
        image_paths = [p.decode("utf-8") for p in f["image_paths"][:]]
        article_ids = [a.decode("utf-8") for a in f["article_ids"][:]]

    if config["safety"]["smoke_test"]:
        n = config["safety"]["smoke_n_samples"]
        image_paths = image_paths[:n]
        article_ids = article_ids[:n]

    batch_size = config["inference"]["batch_size"]
    scores = []
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size), desc=f"  {split}"):
            batch_paths = image_paths[i : i + batch_size]
            feats = preprocessor.extract_features(batch_paths)  # [B, feature_dim]
            feats_t = torch.from_numpy(feats).float().to(device)
            if feats_t.ndim == 1:
                feats_t = feats_t.unsqueeze(0)
            logits = head(feats_t)
            probs = torch.sigmoid(logits).cpu().numpy()
            scores.extend(np.atleast_1d(probs).tolist())

    return {
        "article_ids": np.array(article_ids, dtype="S"),
        "ai_generated_score": np.array(scores, dtype=np.float32),
    }


signal_by_split = {}
for split in CONFIG["data"]["splits"]:
    result = score_coolant_split(split, CONFIG, image_preprocessor, classifier_head, DEVICE)
    if result is not None:
        signal_by_split[split] = result
        print(
            f"  {split:5s}: {len(result['ai_generated_score'])} images scored, "
            f"mean={result['ai_generated_score'].mean():.4f}"
        )

## Step 4: Export Signal Files

Saved as `ai_art_signal_{split}.npz` keyed by `article_ids`, matching row order/count of the
corresponding `coolant_{split}.h5`. Kept separate from the ResNet50 image features — the
consuming code loads and joins by `article_id` rather than requiring a COOLANT architecture change.

In [ ]:
for split, result in signal_by_split.items():
    out_path = CONFIG["paths"]["output_dir"] / f"ai_art_signal_{split}.npz"
    np.savez(
        out_path,
        article_ids=result["article_ids"],
        ai_generated_score=result["ai_generated_score"],
    )
    print(f"Saved: {out_path}")

print(f"\n{'='*55}")
print("Stage 3.6 complete.")
print(f"Signal files: {CONFIG['paths']['output_dir']}")
print(f"Source checkpoint: {checkpoint_path}")

## Note: Follow-up Fusion Into COOLANT

This notebook only produces the scalar `ai_generated_score` signal, joined by `article_id`.
It does **not** modify `ResNetCOOLANT`'s forward pass or `create_coolant_dataloaders`.

Before wiring this into the training loop (e.g. as an extra concatenated feature or an
auxiliary loss term), check whether the signal actually correlates with the fake-news label
on this dataset — e.g. `groupby(label).ai_generated_score.mean()` — since COOLANT news images
are photos, not artwork/GAN images, and the AI-art classifier was trained on a different
image distribution (WikiArt vs GAN art). If the signal doesn't discriminate here, building
the fusion plumbing would be wasted effort.